# Практична робота №9
## Перетворення даних: похідні змінні, дискретизація, масштабування, індикаторне кодування

**Виконав:** Панасюк Дмитро  
**Номер за списком:** 15  
**Варіант:** №5 — **Дніпро**  
**Базова ціна товару:** **810 грн**  
**Базова кількість:** **3 шт.**

**Мета:** створити набір із 10 клієнтів інтернет-магазину та виконати чотири перетворення даних: обчислення похідних змінних, дискретизацію віку, масштабування числових ознак та one-hot кодування міста доставки.

Усі клієнти, доходи й замовлення — **навчальні синтетичні дані**. Використано базові параметри магазину з Практичної №4.

## Підготовка середовища
Імпортуємо `pandas` і `numpy` для створення таблиці, перетворень та числових перевірок.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 2.2.3
numpy: 2.3.5


## Завдання 1. Побудова набору клієнтів

Створимо `DataFrame` з **10 замовлень** та чотирма початковими стовпцями: `вік`, `дохід`, `кількість`, `місто_доставки`. Вік усіх клієнтів — від 18 до 70 років, кількість товарів орієнтується на базові 3 шт., а доставка виконується до **п’яти різних міст**.

Різні масштаби `вік` (роки) та `дохід` (грн/міс.) потрібні для демонстрації нормалізації в Завданні 4.

In [2]:
base_price = 810  # грн/шт.
base_quantity = 3  # шт.
city = "Дніпро"

clients = pd.DataFrame({
    "вік": [20, 22, 25, 29, 30, 35, 43, 50, 57, 69],
    "дохід": [15000, 18000, 22500, 28000, 32000, 39000, 44000, 52000, 67000, 90000],
    "кількість": [3, 2, 4, 3, 5, 2, 3, 4, 1, 3],
    "місто_доставки": [
        "Дніпро", "Київ", "Львів", "Одеса", "Харків",
        "Дніпро", "Київ", "Львів", "Одеса", "Дніпро",
    ],
})

assert clients.shape == (10, 4)
assert clients["вік"].between(18, 70).all()
assert clients["місто_доставки"].nunique() >= 4
assert base_quantity == 3

print("Форма таблиці:", clients.shape)
print("Міста доставки:", clients["місто_доставки"].unique().tolist())
print("Кількість унікальних міст:", clients["місто_доставки"].nunique())
display(clients)

Форма таблиці: (10, 4)
Міста доставки: ['Дніпро', 'Київ', 'Львів', 'Одеса', 'Харків']
Кількість унікальних міст: 5


,вік,дохід,кількість,місто_доставки
0,20,15000,3,Дніпро
1,22,18000,2,Київ
2,25,22500,4,Львів
3,29,28000,3,Одеса
4,30,32000,5,Харків
5,35,39000,2,Дніпро
6,43,44000,3,Київ
7,50,52000,4,Львів
8,57,67000,1,Одеса
9,69,90000,3,Дніпро


## Завдання 2. Похідні змінні

1. `сума_замовлення` = `810 × кількість`, у гривнях.
2. `частка_доходу` = `сума_замовлення / дохід`, безрозмірне число; додатково показуємо його у відсотках для наочності.

Формули застосовуються відразу до цілих стовпців, без циклів.

In [3]:
clients["сума_замовлення"] = base_price * clients["кількість"]
clients["частка_доходу"] = clients["сума_замовлення"] / clients["дохід"]

assert (clients["сума_замовлення"] == clients["кількість"] * 810).all()
assert np.allclose(clients["частка_доходу"], clients["сума_замовлення"] / clients["дохід"])

derived_view = clients[["дохід", "кількість", "сума_замовлення", "частка_доходу"]].copy()
derived_view["частка_доходу_%"] = (derived_view["частка_доходу"] * 100).round(2)
display(derived_view.round({"частка_доходу": 4}))

,дохід,кількість,сума_замовлення,частка_доходу,частка_доходу_%
0,15000,3,2430,0.1620,16.20
1,18000,2,1620,0.0900,9.00
2,22500,4,3240,0.1440,14.40
3,28000,3,2430,0.0868,8.68
4,32000,5,4050,0.1266,12.66
5,39000,2,1620,0.0415,4.15
6,44000,3,2430,0.0552,5.52
7,52000,4,3240,0.0623,6.23
8,67000,1,810,0.0121,1.21
9,90000,3,2430,0.0270,2.70


**Письмове пояснення до Завдання 2.** `частка_доходу` показує не просто суму покупки, а її **навантаження на бюджет клієнта**. Наприклад, замовлення на 2430 грн становить 16,2% доходу 15 000 грн, але лише 2,7% доходу 90 000 грн. Тому ця відносна ознака дає змогу коректніше порівнювати покупки клієнтів із різними доходами, ніж сума замовлення або дохід окремо. Це не показник реальної платоспроможності, адже інші витрати клієнта тут невідомі.

## Завдання 3. Дискретизація віку: `cut()` і `qcut()`

- `pd.cut(..., bins=3)` розбиває **діапазон віку** на три інтервали приблизно однакової довжини (у роках).
- `pd.qcut(..., q=3)` використовує **квантилі**, щоб отримати групи з приблизно рівною кількістю клієнтів. За малого обсягу вибірки та на межах квантилів точна рівність не гарантована.

Виведемо категорії для кожного клієнта та частоти за групами.

In [4]:
clients["вік_група_cut"] = pd.cut(clients["вік"], bins=3)
clients["вік_група_qcut"] = pd.qcut(clients["вік"], q=3)

cut_freq = clients["вік_група_cut"].value_counts().sort_index()
qcut_freq = clients["вік_група_qcut"].value_counts().sort_index()

print("Група для кожного клієнта:")
display(clients[["вік", "вік_група_cut", "вік_група_qcut"]])
print("pd.cut — частоти:")
display(cut_freq.rename("клієнтів").to_frame())
print("pd.qcut — частоти:")
display(qcut_freq.rename("клієнтів").to_frame())

assert cut_freq.sum() == len(clients)
assert qcut_freq.sum() == len(clients)

Група для кожного клієнта:


,вік,вік_група_cut,вік_група_qcut
0,20,"(19.951, 36.333]","(19.999, 29.0]"
1,22,"(19.951, 36.333]","(19.999, 29.0]"
2,25,"(19.951, 36.333]","(19.999, 29.0]"
3,29,"(19.951, 36.333]","(29.0, 43.0]"
4,30,"(19.951, 36.333]","(29.0, 43.0]"
5,35,"(19.951, 36.333]","(29.0, 43.0]"
6,43,"(36.333, 52.667]","(43.0, 69.0]"
7,50,"(36.333, 52.667]","(43.0, 69.0]"
8,57,"(52.667, 69.0]","(43.0, 69.0]"
9,69,"(52.667, 69.0]","(43.0, 69.0]"


pd.cut — частоти:


,клієнтів
вік_група_cut,
"(19.951, 36.333]",6
"(36.333, 52.667]",2
"(52.667, 69.0]",2


pd.qcut — частоти:


,клієнтів
вік_група_qcut,
"(19.999, 29.0]",3
"(29.0, 43.0]",3
"(43.0, 69.0]",4


**Письмове порівняння до Завдання 3.** У `cut` частоти становлять **6, 2, 2**, оскільки шість клієнтів потрапляють у нижню третину діапазону за віком. У `qcut` частоти — **3, 3, 4**: межі інтервалів підлаштовані до розподілу людей, тому інтервали мають різну довжину, але ближчі між собою кількості спостережень.

Це показує різницю підходів: **`cut` орієнтується на ширину інтервалів**, а **`qcut` — на квантилі розподілу**.

## Завдання 4. Масштабування віку й доходу

Застосуємо окремо до кожної числової змінної:

- **Min–max:** $x_{norm} = (x-x_{min})/(x_{max}-x_{min})$ → результат у діапазоні **[0, 1]**.
- **Z-оцінка:** $z = (x-\bar{x})/s$ → середнє близьке до нуля, вибіркове стандартне відхилення **1** (`ddof=1`, як у pandas за замовчуванням).

Порівняємо початкові та масштабовані ознаки поруч.

In [5]:
numeric = clients[["вік", "дохід"]]

minmax = (numeric - numeric.min()) / (numeric.max() - numeric.min())
zscore = (numeric - numeric.mean()) / numeric.std(ddof=1)

# Зберігаємо всі чотири масштабовані стовпці в основному DataFrame.
for col in ["вік", "дохід"]:
    clients[f"{col}_minmax"] = minmax[col]
    clients[f"{col}_z"] = zscore[col]

comparison = pd.concat({
    "Вихідні значення": numeric,
    "Min–max [0, 1]": minmax,
    "Z-оцінка": zscore,
}, axis=1)
print("Перші 5 рядків: первинні дані, min–max, z-оцінка")
display(comparison.head().round(3))

assert minmax.ge(0).all().all() and minmax.le(1).all().all()
assert np.allclose(zscore.mean().to_numpy(), [0, 0], atol=1e-12)
assert np.allclose(zscore.std(ddof=1).to_numpy(), [1, 1], atol=1e-12)

print("Межі після min–max:")
display(pd.DataFrame({"мінімум": minmax.min(), "максимум": minmax.max()}))
print("Перевірка z-оцінок:")
display(pd.DataFrame({"середнє": zscore.mean(), "std (ddof=1)": zscore.std(ddof=1)}).round(5))

Перші 5 рядків: первинні дані, min–max, z-оцінка


Вихідні значення        Min–max [0, 1]        Z-оцінка       
               вік  дохід            вік  дохід      вік  дохід
0               20  15000          0.000  0.000   -1.104 -1.092
1               22  18000          0.041  0.040   -0.981 -0.964
2               25  22500          0.102  0.100   -0.797 -0.774
3               29  28000          0.184  0.173   -0.552 -0.541
4               30  32000          0.204  0.227   -0.491 -0.371

Межі після min–max:


,мінімум,максимум
вік,0.0,1.0
дохід,0.0,1.0


Перевірка z-оцінок:


,середнє,std (ddof=1)
вік,-0.0,1.0
дохід,-0.0,1.0


**Письмове пояснення до Завдання 4.** До масштабування `вік` має значення в десятках років, а `дохід` — у десятках тисяч гривень. В обчисленні евклідової відстані або у регресії з регуляризацією більший числовий масштаб доходу може непропорційно впливати на результат — не тому, що він обов’язково важливіший, а через одиниці вимірювання. Min–max приводить обидві ознаки до `[0, 1]`, а z-оцінка вимірює відхилення від середнього в одиницях стандартного відхилення. Це робить ознаки зіставними за масштабом, хоча не робить їх змістовно однаковими.

## Завдання 5. Індикаторне (one-hot) кодування міст

`місто_доставки` — **номінальна** змінна. Створимо по одному індикаторному стовпцю для кожного унікального міста за допомогою `pd.get_dummies()`, після чого приєднаємо їх до основної таблиці через `pd.concat()`.

Значення **1** означає, що замовлення доставляють у відповідне місто; **0** — що ні.

In [6]:
city_dummies = pd.get_dummies(clients["місто_доставки"], prefix="місто", dtype=int)
clients_encoded = pd.concat([clients, city_dummies], axis=1)

city_count = clients["місто_доставки"].nunique()
print("Унікальних міст:", city_count)
print("Нових індикаторних стовпців:", city_dummies.shape[1])
print("Назви індикаторів:", city_dummies.columns.tolist())
display(clients_encoded[["місто_доставки", *city_dummies.columns]])

assert city_dummies.shape[1] == city_count
assert (city_dummies.sum(axis=1) == 1).all()
assert clients_encoded.shape[0] == 10
print("Перевірку пройдено: для кожного рядка рівно один індикатор дорівнює 1.")

Унікальних міст: 5
Нових індикаторних стовпців: 5
Назви індикаторів: ['місто_Дніпро', 'місто_Київ', 'місто_Львів', 'місто_Одеса', 'місто_Харків']


,місто_доставки,місто_Дніпро,місто_Київ,місто_Львів,місто_Одеса,місто_Харків
0,Дніпро,1,0,0,0,0
1,Київ,0,1,0,0,0
2,Львів,0,0,1,0,0
3,Одеса,0,0,0,1,0
4,Харків,0,0,0,0,1
5,Дніпро,1,0,0,0,0
6,Київ,0,1,0,0,0
7,Львів,0,0,1,0,0
8,Одеса,0,0,0,1,0
9,Дніпро,1,0,0,0,0


Перевірку пройдено: для кожного рядка рівно один індикатор дорівнює 1.


**Письмове пояснення до Завдання 5.** Міста не мають природного порядку: Київ не «більший» і не «менший» за Дніпро як категорія. Кодування числами `1, 2, 3, ...` створило б **штучний порядок і числові відстані** між містами. One-hot кодування уникає цієї проблеми: кожне місто подано незалежним двійковим індикатором. Оскільки в наборі є **5 унікальних міст**, створено **5 нових стовпців**.

## Додаткова підсумкова перевірка

Після перетворень початкова таблиця з 10 клієнтами доповнилась двома похідними змінними, двома віковими категоріями, чотирма масштабованими числовими ознаками та п’ятьма індикаторами міст. Покажемо структуру й перевіримо відсутність пропусків.

In [7]:
print("Підсумкова форма DataFrame:", clients_encoded.shape)
print("Кількість пропусків:", int(clients_encoded.isna().sum().sum()))
print("Усі перетворення виконані для варіанта №5 — Дніпро.")
assert clients_encoded.isna().sum().sum() == 0
assert clients_encoded.shape == (10, 17)
display(clients_encoded.head(3))

Підсумкова форма DataFrame: (10, 17)
Кількість пропусків: 0
Усі перетворення виконані для варіанта №5 — Дніпро.


,вік,дохід,кількість,місто_доставки,сума_замовлення,частка_доходу,вік_група_cut,вік_група_qcut,вік_minmax,вік_z,дохід_minmax,дохід_z,місто_Дніпро,місто_Київ,місто_Львів,місто_Одеса,місто_Харків
0,20,15000,3,Дніпро,2430,0.162,"(19.951, 36.333]","(19.999, 29.0]",0.000000,-1.103651,0.00,-1.091648,1,0,0,0,0
1,22,18000,2,Київ,1620,0.090,"(19.951, 36.333]","(19.999, 29.0]",0.040816,-0.981023,0.04,-0.964466,0,1,0,0,0
2,25,22500,4,Львів,3240,0.144,"(19.951, 36.333]","(19.999, 29.0]",0.102041,-0.797081,0.10,-0.773692,0,0,1,0,0


## Відповіді на контрольні питання

**1. Чим відрізняються `pd.cut()` і `pd.qcut()`?**  
`pd.cut()` розділяє числовий діапазон на інтервали однакової або заданої ширини, тому кількість записів у них може сильно відрізнятися. `pd.qcut()` обирає межі за квантилями, намагаючись розподілити приблизно однакову кількість спостережень між групами. Точна однаковість **не завжди можлива**: наприклад, 10 клієнтів не діляться порівну на 3 групи, а повтори значень на межах ще більше ускладнюють розподіл.

**2. Чому для міста потрібне one-hot кодування?**  
Місто належить до номінальної шкали: назви відрізняються, але не мають рангу чи математичної відстані між ними. Присвоювання довільних чисел `1`, `2`, `3` помилково створило б такі відношення. One-hot подає кожне місто незалежним індикатором `0/1`.

**3. Навіщо масштабувати змінні перед методами, чутливими до масштабу?**  
У методах відстаней, наприклад k-nearest neighbors або k-means, ознака з великими числами може домінувати в метриці. Регресія з регуляризацією також чутлива до того, як масштабовані коефіцієнти та ознаки. Стандартизація або min–max нормалізація усуває штучний вплив одиниць вимірювання: вік у роках і дохід у гривнях отримують порівнянний числовий масштаб.

**4. Чому похідна змінна може бути змістовнішою за вихідні складники?**  
`частка_доходу = сума_замовлення / дохід` враховує розмір покупки **відносно фінансових можливостей клієнта**. Одна й та сама покупка може бути суттєвою витратою для клієнта з меншим доходом і невеликою — для клієнта з більшим. Отже, частка дозволяє порівнювати навантаження покупки на місячний дохід, чого сума замовлення і дохід окремо не показують.

## Висновок

Для інтернет-магазину у **Дніпрі** (варіант №5, базова ціна **810 грн/шт.**) створено набір із 10 замовлень і повністю виконано перетворення з Лекції 9:

- обчислено `сума_замовлення` та `частка_доходу`;
- виконано дискретизацію віку через `pd.cut()` і `pd.qcut()` та порівняно частоти;
- масштабовано вік і дохід методами min–max та z-оцінки;
- створено індикаторні стовпці через `pd.get_dummies()` для п’яти міст доставки.

Отримані дані готові до подальшого аналізу; числові перевірки підтверджують правильність формул, масштабування та кодування.